In [ ]:
import pandas as pd
import numpy as np


## %WEIGHTED ILI in influenza cases
The term %WEIGHTED ILI refers to the percentage of visits for influenza-like illness reported by sentinel providers, such as healthcare providers in the U.S. Outpatient Influenza-like Illness Surveillance Network (ILINet). This data is collected weekly and weighted based on state population to provide a more accurate representation of the prevalence of ILI across different age groups. The data is collected through electronic records and is used to monitor seasonal variations of influenza activity and identify the best patients for virological surveillance

In [ ]:
df=pd.read_csv("/content/ILINet_region.csv")

In [ ]:
df.shape


(14700, 15)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14700 entries, 0 to 14699
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   REGION TYPE        14700 non-null  object 
 1   REGION             14700 non-null  object 
 2   YEAR               14700 non-null  int64  
 3   WEEK               14700 non-null  int64  
 4   % WEIGHTED ILI     14700 non-null  float64
 5   %UNWEIGHTED ILI    14700 non-null  float64
 6   AGE 0-4            14700 non-null  int64  
 7   AGE 25-49          9380 non-null   float64
 8   AGE 25-64          6270 non-null   float64
 9   AGE 5-24           14700 non-null  int64  
 10  AGE 50-64          9380 non-null   float64
 11  AGE 65             14700 non-null  int64  
 12  ILITOTAL           14700 non-null  int64  
 13  NUM. OF PROVIDERS  14700 non-null  int64  
 14  TOTAL PATIENTS     14700 non-null  int64  
dtypes: float64(5), int64(8), object(2)
memory usage: 1.7+ MB


In [ ]:
df.head()

,REGION TYPE,REGION,YEAR,WEEK,% WEIGHTED ILI,%UNWEIGHTED ILI,AGE 0-4,AGE 25-49,AGE 25-64,AGE 5-24,AGE 50-64,AGE 65,ILITOTAL,NUM. OF PROVIDERS,TOTAL PATIENTS
0,HHS Regions,Region 1,1997,40,0.498535,0.623848,15,NaN,7.0,22,NaN,0,44,32,7053
1,HHS Regions,Region 2,1997,40,0.374963,0.384615,0,NaN,3.0,0,NaN,0,3,7,780
2,HHS Regions,Region 3,1997,40,1.354280,1.341720,6,NaN,7.0,15,NaN,4,32,16,2385
3,HHS Regions,Region 4,1997,40,0.400338,0.450010,12,NaN,23.0,11,NaN,0,46,29,10222
4,HHS Regions,Region 5,1997,40,1.229260,0.901266,31,NaN,24.0,30,NaN,4,89,49,9875


In [ ]:
df.isnull().sum()
df.isnull().mean() * 100


,0
REGION TYPE,0.000000
REGION,0.000000
YEAR,0.000000
WEEK,0.000000
% WEIGHTED ILI,0.000000
%UNWEIGHTED ILI,0.000000
AGE 0-4,0.000000
AGE 25-49,36.190476
AGE 25-64,57.346939
AGE 5-24,0.000000


In [ ]:
df = df.sort_values(by=["REGION TYPE", "REGION", "YEAR", "WEEK"])


In [ ]:
df.head()

,REGION TYPE,REGION,YEAR,WEEK,% WEIGHTED ILI,%UNWEIGHTED ILI,AGE 0-4,AGE 25-49,AGE 25-64,AGE 5-24,AGE 50-64,AGE 65,ILITOTAL,NUM. OF PROVIDERS,TOTAL PATIENTS
0,HHS Regions,Region 1,1997,40,0.498535,0.623848,15,NaN,7.0,22,NaN,0,44,32,7053
10,HHS Regions,Region 1,1997,41,0.642669,0.815801,14,NaN,14.0,29,NaN,0,57,29,6987
20,HHS Regions,Region 1,1997,42,2.899080,1.225840,35,NaN,15.0,35,NaN,0,85,40,6934
30,HHS Regions,Region 1,1997,43,4.812500,1.621690,41,NaN,25.0,59,NaN,0,125,44,7708
40,HHS Regions,Region 1,1997,44,1.371360,0.780165,22,NaN,14.0,34,NaN,3,73,44,9357


In [ ]:
age_cols = ["AGE 25-49", "AGE 25-64", "AGE 50-64"]

df[age_cols] = (
    df
    .groupby(["REGION TYPE", "REGION"])[age_cols]
    .transform(
        lambda x: x.ffill()
                   .rolling(window=3, min_periods=1)
                   .mean()
                   .bfill()
    )
)


In [ ]:
df[age_cols].isnull().sum()


,0
AGE 25-49,0
AGE 25-64,0
AGE 50-64,0


In [ ]:
df.head()

,REGION TYPE,REGION,YEAR,WEEK,% WEIGHTED ILI,%UNWEIGHTED ILI,AGE 0-4,AGE 25-49,AGE 25-64,AGE 5-24,AGE 50-64,AGE 65,ILITOTAL,NUM. OF PROVIDERS,TOTAL PATIENTS
0,HHS Regions,Region 1,1997,40,0.498535,0.623848,15,0.0,7.0,22,0.0,0,44,32,7053
10,HHS Regions,Region 1,1997,41,0.642669,0.815801,14,0.0,10.5,29,0.0,0,57,29,6987
20,HHS Regions,Region 1,1997,42,2.899080,1.225840,35,0.0,12.0,35,0.0,0,85,40,6934
30,HHS Regions,Region 1,1997,43,4.812500,1.621690,41,0.0,18.0,59,0.0,0,125,44,7708
40,HHS Regions,Region 1,1997,44,1.371360,0.780165,22,0.0,18.0,34,0.0,3,73,44,9357


In [ ]:
age_cols = ["AGE 0-4", "AGE 5-24", "AGE 25-49", "AGE 50-64", "AGE 65"]

for col in age_cols:
    print(f"\n🔹 {col}")

    # get unique non-null values, sort them, print first 10
    unique_sorted = sorted(df[col].dropna().unique())

    print("First 20 sorted values:", unique_sorted[:20])
    print("Total unique values:", len(unique_sorted))



🔹 AGE 0-4
First 20 sorted values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19)]
Total unique values: 2173

🔹 AGE 5-24
First 20 sorted values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19)]
Total unique values: 2692

🔹 AGE 25-49
First 20 sorted values: [np.float64(0.0), np.float64(0.3333333333333333), np.float64(0.6666666666666666), np.float64(1.0), np.float64(1.3333333333333333), np.float64(1.6666666666666667), np.float64(2.0), np.float64(2.3333333333333335), np.float64(2.6666666666666665), np.float64(3.0), np.float64(3.3333333333333335), np.fl

In [ ]:
age_cols = ["AGE 0-4", "AGE 5-24", "AGE 25-49", "AGE 50-64", "AGE 65"]

for col in age_cols:
    print(f"\n🔹 {col}")

    vc = (
        df[col]
        .value_counts(dropna=False)   # include NaN if any
        .sort_index()                 # sort by value (0,1,2,…)
        .head(10)                     # print only first 10 values
    )

    print(vc)



🔹 AGE 0-4
AGE 0-4
0    1125
1     123
2     135
3     124
4     158
5     101
6     107
7      98
8      91
9      74
Name: count, dtype: int64

🔹 AGE 5-24
AGE 5-24
0    1074
1      63
2      79
3      84
4      81
5      83
6      73
7      92
8      97
9      68
Name: count, dtype: int64

🔹 AGE 25-49
AGE 25-49
0.000000    6270
0.333333       1
0.666667       2
1.000000       4
1.333333       6
1.666667       3
2.000000       8
2.333333      10
2.666667      11
3.000000      10
Name: count, dtype: int64

🔹 AGE 50-64
AGE 50-64
0.000000    6272
0.333333       5
0.666667       9
1.000000      12
1.333333      19
1.666667      16
2.000000      16
2.333333      27
2.666667      25
3.000000      34
Name: count, dtype: int64

🔹 AGE 65
AGE 65
0    1513
1     448
2     431
3     315
4     309
5     231
6     207
7     212
8     179
9     169
Name: count, dtype: int64


In [ ]:
df.isnull().sum()

,0
REGION TYPE,0
REGION,0
YEAR,0
WEEK,0
% WEIGHTED ILI,0
%UNWEIGHTED ILI,0
AGE 0-4,0
AGE 25-49,0
AGE 25-64,0
AGE 5-24,0


In [ ]:
rename_map = {
    "REGION TYPE": "region_type",
    "REGION": "region",
    "YEAR": "year",
    "WEEK": "week",
    "% WEIGHTED ILI": "ili_weighted_pct",
    "%UNWEIGHTED ILI": "ili_unweighted_pct",
    "AGE 0-4": "cases_age_0_4",
    "AGE 5-24": "cases_age_5_24",
    "AGE 25-49": "cases_age_25_49",
    "AGE 50-64": "cases_age_50_64",
    "AGE 65": "cases_age_65_plus",
    "ILITOTAL": "ili_total_cases",
    "NUM. OF PROVIDERS": "num_reporting_providers",
    "TOTAL PATIENTS": "total_patients_seen"
}

df.rename(columns=rename_map, inplace=True)


In [ ]:
df.head()

,region_type,region,year,week,ili_weighted_pct,ili_unweighted_pct,cases_age_0_4,cases_age_25_49,AGE 25-64,cases_age_5_24,cases_age_50_64,cases_age_65_plus,ili_total_cases,num_reporting_providers,total_patients_seen
0,HHS Regions,Region 1,1997,40,0.498535,0.623848,15,0.0,7.0,22,0.0,0,44,32,7053
10,HHS Regions,Region 1,1997,41,0.642669,0.815801,14,0.0,10.5,29,0.0,0,57,29,6987
20,HHS Regions,Region 1,1997,42,2.899080,1.225840,35,0.0,12.0,35,0.0,0,85,40,6934
30,HHS Regions,Region 1,1997,43,4.812500,1.621690,41,0.0,18.0,59,0.0,0,125,44,7708
40,HHS Regions,Region 1,1997,44,1.371360,0.780165,22,0.0,18.0,34,0.0,3,73,44,9357


In [ ]:
# define output path
output_path = "influenza_cleaned_dataset.csv"

# save dataframe
df.to_csv(output_path, index=False)

print(f"✅ Cleaned dataset saved successfully as: {output_path}")


✅ Cleaned dataset saved successfully as: influenza_cleaned_dataset.csv


In [ ]:
df2=pd.read_csv("/content/ICL_NREVSS_Public_Health_Labs.csv")

In [ ]:
df2.shape


(5300, 13)

In [ ]:
df2.head()

,REGION TYPE,REGION,YEAR,WEEK,TOTAL SPECIMENS,A (2009 H1N1),A (H3),A (Subtyping not Performed),B,BVic,BYam,H3N2v,A (H5)
0,HHS Regions,Region 1,2015,40,39,0,5,0,0,0,0,0,0
1,HHS Regions,Region 2,2015,40,56,1,4,0,1,0,0,0,0
2,HHS Regions,Region 3,2015,40,132,1,3,0,0,0,0,0,0
3,HHS Regions,Region 4,2015,40,83,0,5,0,1,0,0,0,0
4,HHS Regions,Region 5,2015,40,218,2,7,0,0,0,1,0,0


In [ ]:
df2.isnull().sum()
df2.isnull().mean() * 100


,0
REGION TYPE,0.0
REGION,0.0
YEAR,0.0
WEEK,0.0
TOTAL SPECIMENS,0.0
A (2009 H1N1),0.0
A (H3),0.0
A (Subtyping not Performed),0.0
B,0.0
BVic,0.0


In [ ]:
df2.head(20)

,REGION TYPE,REGION,YEAR,WEEK,TOTAL SPECIMENS,A (2009 H1N1),A (H3),A (Subtyping not Performed),B,BVic,BYam,H3N2v,A (H5)
0,HHS Regions,Region 1,2015,40,39,0,5,0,0,0,0,0,0
1,HHS Regions,Region 2,2015,40,56,1,4,0,1,0,0,0,0
2,HHS Regions,Region 3,2015,40,132,1,3,0,0,0,0,0,0
3,HHS Regions,Region 4,2015,40,83,0,5,0,1,0,0,0,0
4,HHS Regions,Region 5,2015,40,218,2,7,0,0,0,1,0,0
5,HHS Regions,Region 6,2015,40,97,0,2,0,0,0,0,0,0
6,HHS Regions,Region 7,2015,40,36,0,2,0,0,0,0,0,0
7,HHS Regions,Region 8,2015,40,71,0,2,0,0,0,0,0,0
8,HHS Regions,Region 9,2015,40,273,0,22,2,8,0,0,0,0
9,HHS Regions,Region 10,2015,40,134,0,13,0,0,0,0,0,0


In [ ]:
rename_map_df2 = {
    "REGION TYPE": "region_type",
    "REGION": "region",
    "YEAR": "year",
    "WEEK": "week",
    "TOTAL SPECIMENS": "total_specimens_tested",
    "A (H3)": "cases_a_h3",
    "A (2009 H1N1)": "cases_a_h1n1_pdm09",
    "A (Subtyping not Performed)": "cases_a_not_subtyped",
    "A (H5)": "cases_a_h5",
    "B": "cases_b_unspecified",
    "BVic": "cases_b_victoria",
    "BYam": "cases_b_yamagata",
    "H3N2v": "cases_h3n2_variant"
}

df2.rename(columns=rename_map_df2, inplace=True)


In [ ]:
# define output path
output_path = "public_health_lab_cleaned_dataset.csv"

# save dataframe
df2.to_csv(output_path, index=False)

print(f"✅ Cleaned dataset saved successfully as: {output_path}")


✅ Cleaned dataset saved successfully as: public_health_lab_cleaned_dataset.csv


In [ ]:
df2['year'].value_counts()

,count
year,
2020,530
2017,520
2016,520
2022,520
2018,520
2019,520
2021,520
2024,520
2023,520


In [ ]:
df2.head()

,region_type,region,year,week,total_specimens_tested,cases_a_h1n1_pdm09,cases_a_h3,cases_a_not_subtyped,cases_b_unspecified,cases_b_victoria,cases_b_yamagata,cases_h3n2_variant,cases_a_h5
0,HHS Regions,Region 1,2015,40,39,0,5,0,0,0,0,0,0
1,HHS Regions,Region 2,2015,40,56,1,4,0,1,0,0,0,0
2,HHS Regions,Region 3,2015,40,132,1,3,0,0,0,0,0,0
3,HHS Regions,Region 4,2015,40,83,0,5,0,1,0,0,0,0
4,HHS Regions,Region 5,2015,40,218,2,7,0,0,0,1,0,0


In [ ]:
df2["total_a"] = (
    df2["cases_a_h1n1_pdm09"] +
    df2["cases_a_h3"] +
    df2["cases_a_h5"] +
    df2["cases_a_not_subtyped"]
)

df2["total_b"] = (
    df2["cases_b_victoria"] +
    df2["cases_b_yamagata"] +
    df2["cases_b_unspecified"]
)

df2["percent_positive"] = (df2["total_a"] + df2["total_b"]) / df2["total_specimens_tested"] * 100


In [ ]:
df2.head()

,region_type,region,year,week,total_specimens_tested,cases_a_h1n1_pdm09,cases_a_h3,cases_a_not_subtyped,cases_b_unspecified,cases_b_victoria,cases_b_yamagata,cases_h3n2_variant,cases_a_h5,total_a,total_b,percent_positive
0,HHS Regions,Region 1,2015,40,39,0,5,0,0,0,0,0,0,5,0,12.820513
1,HHS Regions,Region 2,2015,40,56,1,4,0,1,0,0,0,0,5,1,10.714286
2,HHS Regions,Region 3,2015,40,132,1,3,0,0,0,0,0,0,4,0,3.030303
3,HHS Regions,Region 4,2015,40,83,0,5,0,1,0,0,0,0,5,1,7.228916
4,HHS Regions,Region 5,2015,40,218,2,7,0,0,0,1,0,0,9,1,4.587156


In [ ]:
# define output path
output_path = "public_health_lab_cleaned_dataset.csv"

# save dataframe
df2.to_csv(output_path, index=False)

print(f"✅ Cleaned dataset saved successfully as: {output_path}")


✅ Cleaned dataset saved successfully as: public_health_lab_cleaned_dataset.csv


In [ ]:
df2.isnull().sum()

,0
region_type,0
region,0
year,0
week,0
total_specimens_tested,0
cases_a_h1n1_pdm09,0
cases_a_h3,0
cases_a_not_subtyped,0
cases_b_unspecified,0
cases_b_victoria,0


In [ ]:
df2['percent_positive'].value_counts()

,count
percent_positive,
0.000000,736
25.000000,34
33.333333,33
20.000000,31
50.000000,28
...,...
18.032787,1
70.967742,1
12.857143,1


In [ ]:
df2[df2['percent_positive'].isna()]


,region_type,region,year,week,total_specimens_tested,cases_a_h1n1_pdm09,cases_a_h3,cases_a_not_subtyped,cases_b_unspecified,cases_b_victoria,cases_b_yamagata,cases_h3n2_variant,cases_a_h5,total_a,total_b,percent_positive
360,HHS Regions,Region 1,2016,24,0,0,0,0,0,0,0,0,0,0,0,NaN
370,HHS Regions,Region 1,2016,25,0,0,0,0,0,0,0,0,0,0,0,NaN
400,HHS Regions,Region 1,2016,28,0,0,0,0,0,0,0,0,0,0,0,NaN
410,HHS Regions,Region 1,2016,29,0,0,0,0,0,0,0,0,0,0,0,NaN
430,HHS Regions,Region 1,2016,31,0,0,0,0,0,0,0,0,0,0,0,NaN
910,HHS Regions,Region 1,2017,27,0,0,0,0,0,0,0,0,0,0,0,NaN
930,HHS Regions,Region 1,2017,29,0,0,0,0,0,0,0,0,0,0,0,NaN
950,HHS Regions,Region 1,2017,31,0,0,0,0,0,0,0,0,0,0,0,NaN
1420,HHS Regions,Region 1,2018,26,0,0,0,0,0,0,0,0,0,0,0,NaN
1436,HHS Regions,Region 7,2018,27,0,0,0,0,0,0,0,0,0,0,0,NaN


In [ ]:
import numpy as np

df2["percent_positive"] = np.where(
    df2["total_specimens_tested"] > 0,
    (df2["total_a"] + df2["total_b"]) / df2["total_specimens_tested"] * 100,
    np.nan
)


In [ ]:
df2.isnull().sum()

,0
region_type,0
region,0
year,0
week,0
total_specimens_tested,0
cases_a_h1n1_pdm09,0
cases_a_h3,0
cases_a_not_subtyped,0
cases_b_unspecified,0
cases_b_victoria,0


In [ ]:
missing_percent = (df2["percent_positive"].isna().sum() / len(df2)) * 100
missing_percent


np.float64(0.4528301886792453)

In [ ]:
print(f"Missing % in percent_positive: {missing_percent:.2f}%")


Missing % in percent_positive: 0.45%


In [ ]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5300 entries, 0 to 5299
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   region_type             5300 non-null   object 
 1   region                  5300 non-null   object 
 2   year                    5300 non-null   int64  
 3   week                    5300 non-null   int64  
 4   total_specimens_tested  5300 non-null   int64  
 5   cases_a_h1n1_pdm09      5300 non-null   int64  
 6   cases_a_h3              5300 non-null   int64  
 7   cases_a_not_subtyped    5300 non-null   int64  
 8   cases_b_unspecified     5300 non-null   int64  
 9   cases_b_victoria        5300 non-null   int64  
 10  cases_b_yamagata        5300 non-null   int64  
 11  cases_h3n2_variant      5300 non-null   int64  
 12  cases_a_h5              5300 non-null   int64  
 13  total_a                 5300 non-null   int64  
 14  total_b                 5300 non-null   

In [ ]:
df2 = df2.sort_values(
    by=["region_type", "region", "year", "week"]
)


In [ ]:
df2["percent_positive"] = (
    df2
    .groupby(["region_type", "region"])["percent_positive"]
    .ffill()
)


In [ ]:
df2.head()

,region_type,region,year,week,total_specimens_tested,cases_a_h1n1_pdm09,cases_a_h3,cases_a_not_subtyped,cases_b_unspecified,cases_b_victoria,cases_b_yamagata,cases_h3n2_variant,cases_a_h5,total_a,total_b,percent_positive
0,HHS Regions,Region 1,2015,40,39,0,5,0,0,0,0,0,0,5,0,12.820513
10,HHS Regions,Region 1,2015,41,39,2,3,0,0,0,0,0,0,5,0,12.820513
20,HHS Regions,Region 1,2015,42,27,0,3,0,0,0,0,0,0,3,0,11.111111
30,HHS Regions,Region 1,2015,43,35,0,1,0,0,0,0,0,0,1,0,2.857143
40,HHS Regions,Region 1,2015,44,47,0,0,0,0,0,0,0,0,0,0,0.000000


In [ ]:
df2["percent_positive"] = df2["percent_positive"].fillna(0)


In [ ]:
df2["percent_positive"].isna().sum()


np.int64(0)

In [ ]:
# define output path
output_path = "public_health_labs_cleaned.csv"

# save dataframe
df2.to_csv(output_path, index=False)

print(f"✅ Cleaned dataset saved successfully as: {output_path}")


✅ Cleaned dataset saved successfully as: public_health_labs_cleaned.csv


In [ ]:
df3=pd.read_csv("/content/ICL_NREVSS_Clinical_Labs.csv")

In [ ]:
df3.shape


(5300, 10)

In [ ]:
df3.head()

,REGION TYPE,REGION,YEAR,WEEK,TOTAL SPECIMENS,TOTAL A,TOTAL B,PERCENT POSITIVE,PERCENT A,PERCENT B
0,HHS Regions,Region 1,2015,40,693,2,3,0.721501,0.288600,0.432900
1,HHS Regions,Region 2,2015,40,1220,5,0,0.409836,0.409836,0.000000
2,HHS Regions,Region 3,2015,40,896,0,1,0.111607,0.000000,0.111607
3,HHS Regions,Region 4,2015,40,2486,24,16,1.609010,0.965406,0.643604
4,HHS Regions,Region 5,2015,40,2138,14,3,0.795136,0.654818,0.140318


In [ ]:
df3.isnull().sum()
df3.isnull().mean() * 100


,0
REGION TYPE,0.0
REGION,0.0
YEAR,0.0
WEEK,0.0
TOTAL SPECIMENS,0.0
TOTAL A,0.0
TOTAL B,0.0
PERCENT POSITIVE,0.0
PERCENT A,0.0
PERCENT B,0.0


In [ ]:
rename_map_df3 = {
    "REGION TYPE": "region_type",
    "REGION": "region",
    "YEAR": "year",
    "WEEK": "week",
    "TOTAL SPECIMENS": "total_specimens_tested",
    "TOTAL A": "total_influenza_a_cases",
    "TOTAL B": "total_influenza_b_cases",
    "PERCENT POSITIVE": "percent_positive_overall",
    "PERCENT A": "percent_positive_a",
    "PERCENT B": "percent_positive_b"
}

df3.rename(columns=rename_map_df3, inplace=True)


In [ ]:
df3.head()

,region_type,region,year,week,total_specimens_tested,total_influenza_a_cases,total_influenza_b_cases,percent_positive_overall,percent_positive_a,percent_positive_b
0,HHS Regions,Region 1,2015,40,693,2,3,0.721501,0.288600,0.432900
1,HHS Regions,Region 2,2015,40,1220,5,0,0.409836,0.409836,0.000000
2,HHS Regions,Region 3,2015,40,896,0,1,0.111607,0.000000,0.111607
3,HHS Regions,Region 4,2015,40,2486,24,16,1.609010,0.965406,0.643604
4,HHS Regions,Region 5,2015,40,2138,14,3,0.795136,0.654818,0.140318


In [ ]:
# define output path
output_path = "clinical_labs_cleaned_dataset.csv"

# save dataframe
df3.to_csv(output_path, index=False)

print(f"✅ Cleaned dataset saved successfully as: {output_path}")


✅ Cleaned dataset saved successfully as: clinical_labs_cleaned_dataset.csv


✅ STEP 1: Trim ILINet to 2015 onward

In [ ]:
ilin_2015 = df[df["year"] >= 2015].copy()


✅ STEP 2: Trim df3 to 2015 onward (safety check)

In [ ]:
df3_2015 = df3[df3["year"] >= 2015].copy()


✅ STEP 3: Sort both datasets (CRITICAL for time-series)

In [ ]:
sort_cols = ["region_type", "region", "year", "week"]

ilin_2015 = ilin_2015.sort_values(sort_cols)
df3_2015  = df3_2015.sort_values(sort_cols)


✅ STEP 4: Merge to create MODELING DATASET

In [ ]:
model_df = pd.merge(
    ilin_2015,
    df3_2015,
    on=["region_type", "region", "year", "week"],
    how="inner"
)


In [ ]:
print("Modeling dataset shape:", model_df.shape)
print("\nMissing values (%):")
print(model_df.isnull().mean() * 100)

model_df.head()


Modeling dataset shape: (5300, 21)

Missing values (%):
region_type                 0.0
region                      0.0
year                        0.0
week                        0.0
ili_weighted_pct            0.0
ili_unweighted_pct          0.0
cases_age_0_4               0.0
cases_age_25_49             0.0
AGE 25-64                   0.0
cases_age_5_24              0.0
cases_age_50_64             0.0
cases_age_65_plus           0.0
ili_total_cases             0.0
num_reporting_providers     0.0
total_patients_seen         0.0
total_specimens_tested      0.0
total_influenza_a_cases     0.0
total_influenza_b_cases     0.0
percent_positive_overall    0.0
percent_positive_a          0.0
percent_positive_b          0.0
dtype: float64


,region_type,region,year,week,ili_weighted_pct,ili_unweighted_pct,cases_age_0_4,cases_age_25_49,AGE 25-64,cases_age_5_24,...,cases_age_65_plus,ili_total_cases,num_reporting_providers,total_patients_seen,total_specimens_tested,total_influenza_a_cases,total_influenza_b_cases,percent_positive_overall,percent_positive_a,percent_positive_b
0,HHS Regions,Region 1,2015,40,0.743302,0.684364,103,30.333333,62.0,133,...,13,322,134,47051,693,2,3,0.721501,0.288600,0.432900
1,HHS Regions,Region 1,2015,41,0.613241,0.575962,67,36.000000,62.0,123,...,12,257,134,44621,752,11,4,1.994680,1.462770,0.531915
2,HHS Regions,Region 1,2015,42,0.721747,0.667951,83,45.666667,62.0,153,...,13,326,136,48806,708,4,3,0.988701,0.564972,0.423729
3,HHS Regions,Region 1,2015,43,0.767136,0.615777,69,37.666667,62.0,148,...,21,290,133,47095,716,1,1,0.279330,0.139665,0.139665
4,HHS Regions,Region 1,2015,44,0.923337,0.781094,88,46.666667,62.0,200,...,15,391,138,50058,781,0,1,0.128041,0.000000,0.128041


In [ ]:
model_df.to_csv("influenza_modeling_dataset_2015_present.csv", index=False)


In [ ]:
df4=pd.read_csv("/content/ICL_NREVSS_Combined_prior_to_2015_16.csv")

In [ ]:
df4.shape


(9400, 14)

In [ ]:
df4.head()

,REGION TYPE,REGION,YEAR,WEEK,TOTAL SPECIMENS,PERCENT POSITIVE,A (2009 H1N1),A (H1),A (H3),A (Subtyping not Performed),A (Unable to Subtype),B,H3N2v,A (H5)
0,HHS Regions,Region 1,1997,40,51,0.0,0,0,0,0,0,0,0,0
1,HHS Regions,Region 2,1997,40,155,0.0,0,0,0,0,0,0,0,0
2,HHS Regions,Region 3,1997,40,127,0.0,0,0,0,0,0,0,0,0
3,HHS Regions,Region 4,1997,40,183,0.0,0,0,0,0,0,0,0,0
4,HHS Regions,Region 5,1997,40,204,0.0,0,0,0,0,0,0,0,0


In [ ]:
df4.isnull().sum()
df4.isnull().mean() * 100


,0
REGION TYPE,0.0
REGION,0.0
YEAR,0.0
WEEK,0.0
TOTAL SPECIMENS,0.0
PERCENT POSITIVE,0.0
A (2009 H1N1),0.0
A (H1),0.0
A (H3),0.0
A (Subtyping not Performed),0.0


In [ ]:
df4['YEAR'].value_counts().sort_index()


,count
YEAR,
1997,140
1998,520
1999,520
2000,520
2001,520
2002,520
2003,530
2004,520
2005,520


In [ ]:
rename_map_df4 = {
    "REGION TYPE": "region_type",
    "REGION": "region",
    "YEAR": "year",
    "WEEK": "week",
    "TOTAL SPECIMENS": "total_specimens_tested",
    "PERCENT POSITIVE": "percent_positive_overall",
    "A (2009 H1N1)": "cases_a_h1n1_2009",
    "A (H1)": "cases_a_h1",
    "A (H3)": "cases_a_h3",
    "A (Subtyping not Performed)": "cases_a_not_subtyped",
    "A (Unable to Subtype)": "cases_a_unable_to_subtype",
    "B": "cases_b_total",
    "H3N2v": "cases_h3n2_variant",
    "A (H5)": "cases_a_h5"
}

df4.rename(columns=rename_map_df4, inplace=True)


In [ ]:
df4.to_csv("pre_2015_clincal_labs.csv", index=False)


In [ ]:
df5=pd.read_csv("/content/VirusViewBySeason.csv")

In [ ]:
df5.head()

,Season,Virus,0-4 yr,5-24 yr,25-64 yr,65+ yr
0,2025-26,A (H3),410,1467,781,487
1,2025-26,A (H1N1)pdm09,93,179,273,234
2,2025-26,A (Subtyping not Performed),34,98,73,66
3,2025-26,B (Victoria Lineage),4,31,8,1
4,2025-26,B (Lineage Unspecified),15,109,31,7


In [ ]:
df5['Season'].value_counts().sort_index()


,count
Season,
1997-98,4
1998-99,4
1999-00,4
2000-01,4
2001-02,4
2002-03,4
2003-04,4
2004-05,4
2005-06,4


In [ ]:
df5.shape

(147, 6)

In [ ]:
df5 = df5.rename(columns={
    "Season": "season",
    "Virus": "virus",
    "0-4 yr": "cases_age_0_4",
    "5-24 yr": "cases_age_5_24",
    "25-64 yr": "cases_age_25_64",
    "65+ yr": "cases_age_65_plus"
})


In [ ]:
age_cols = [
    "cases_age_0_4",
    "cases_age_5_24",
    "cases_age_25_64",
    "cases_age_65_plus"
]

df5["total_cases"] = df5[age_cols].sum(axis=1)


In [ ]:
print(df5.columns.tolist())


['season', ' Virus', 'cases_age_0_4', 'cases_age_5_24', 'cases_age_25_64', 'cases_age_65_plus', 'total_cases']


In [ ]:
df5.head()

,season,Virus,cases_age_0_4,cases_age_5_24,cases_age_25_64,cases_age_65_plus,total_cases
0,2025-26,A (H3),410,1467,781,487,3145
1,2025-26,A (H1N1)pdm09,93,179,273,234,779
2,2025-26,A (Subtyping not Performed),34,98,73,66,271
3,2025-26,B (Victoria Lineage),4,31,8,1,44
4,2025-26,B (Lineage Unspecified),15,109,31,7,162


In [ ]:
virus_dominance = (
    df5.groupby(["season", " Virus"])["total_cases"]
    .sum()
    .reset_index()
    .sort_values(["season", "total_cases"], ascending=[True, False])
)


In [ ]:
df5 = df5.rename(columns={' Virus': 'virus'})


In [ ]:
df5.head()

,season,virus,cases_age_0_4,cases_age_5_24,cases_age_25_64,cases_age_65_plus,total_cases
0,2025-26,A (H3),410,1467,781,487,3145
1,2025-26,A (H1N1)pdm09,93,179,273,234,779
2,2025-26,A (Subtyping not Performed),34,98,73,66,271
3,2025-26,B (Victoria Lineage),4,31,8,1,44
4,2025-26,B (Lineage Unspecified),15,109,31,7,162


In [ ]:
df5["age_sum"] = (
    df5["cases_age_0_4"]
  + df5["cases_age_5_24"]
  + df5["cases_age_25_64"]
  + df5["cases_age_65_plus"]
)

df5["diff"] = df5["total_cases"] - df5["age_sum"]

df5[["season", "virus", "total_cases", "age_sum", "diff"]].head()


,season,virus,total_cases,age_sum,diff
0,2025-26,A (H3),3145,3145,0
1,2025-26,A (H1N1)pdm09,779,779,0
2,2025-26,A (Subtyping not Performed),271,271,0
3,2025-26,B (Victoria Lineage),44,44,0
4,2025-26,B (Lineage Unspecified),162,162,0


In [ ]:
age_cols = [
    "cases_age_0_4",
    "cases_age_5_24",
    "cases_age_25_64",
    "cases_age_65_plus"
]

age_burden = (
    df5.groupby("virus")[age_cols]
       .sum()
       .reset_index()
)

age_burden


,virus,cases_age_0_4,cases_age_5_24,cases_age_25_64,cases_age_65_plus
0,A (H1),2995,8479,4284,262
1,A (H1N1)pdm09,31089,87046,97949,36193
2,A (H3),32069,87845,86214,78040
3,A (Subtyping not Performed),5571,15132,16096,8436
4,A (Unable to Subtype),119,337,343,27
5,B (Lineage Unspecified),7664,30080,16179,5517
6,B (Victoria Lineage),4206,15797,9463,1749
7,B (Yamagata Lineage),1194,5381,6962,6308
8,H3N2v,120,239,12,6


In [ ]:
for col in age_cols:
    df5[col + "_pct"] = df5[col] / df5["total_cases"] * 100

df5[[
    "season", "virus",
    "cases_age_0_4_pct",
    "cases_age_5_24_pct",
    "cases_age_25_64_pct",
    "cases_age_65_plus_pct"
]].head()

,season,virus,cases_age_0_4_pct,cases_age_5_24_pct,cases_age_25_64_pct,cases_age_65_plus_pct
0,2025-26,A (H3),13.036566,46.645469,24.833068,15.484897
1,2025-26,A (H1N1)pdm09,11.938383,22.978177,35.044929,30.038511
2,2025-26,A (Subtyping not Performed),12.546125,36.162362,26.937269,24.354244
3,2025-26,B (Victoria Lineage),9.090909,70.454545,18.181818,2.272727
4,2025-26,B (Lineage Unspecified),9.259259,67.283951,19.135802,4.320988


In [ ]:
season_severity = (
    df5.groupby("season")["total_cases"]
       .sum()
       .reset_index()
       .sort_values("total_cases", ascending=False)
)

season_severity


,season,total_cases
26,2024-25,82118
10,2008-09,70939
19,2017-18,48295
11,2009-10,43593
20,2018-19,41730
21,2019-20,41447
16,2014-15,40318
14,2012-13,39136
18,2016-17,38641
25,2023-24,38194


In [ ]:
df5["elderly_ratio"] = df5["cases_age_65_plus"] / df5["total_cases"]
df5["child_ratio"]   = df5["cases_age_0_4"] / df5["total_cases"]
df5["working_ratio"] = df5["cases_age_25_64"] / df5["total_cases"]


In [ ]:
df5.head()

,season,virus,cases_age_0_4,cases_age_5_24,cases_age_25_64,cases_age_65_plus,total_cases,age_sum,diff,cases_age_0_4_pct,cases_age_5_24_pct,cases_age_25_64_pct,cases_age_65_plus_pct,elderly_ratio,child_ratio,working_ratio
0,2025-26,A (H3),410,1467,781,487,3145,3145,0,13.036566,46.645469,24.833068,15.484897,0.154849,0.130366,0.248331
1,2025-26,A (H1N1)pdm09,93,179,273,234,779,779,0,11.938383,22.978177,35.044929,30.038511,0.300385,0.119384,0.350449
2,2025-26,A (Subtyping not Performed),34,98,73,66,271,271,0,12.546125,36.162362,26.937269,24.354244,0.243542,0.125461,0.269373
3,2025-26,B (Victoria Lineage),4,31,8,1,44,44,0,9.090909,70.454545,18.181818,2.272727,0.022727,0.090909,0.181818
4,2025-26,B (Lineage Unspecified),15,109,31,7,162,162,0,9.259259,67.283951,19.135802,4.320988,0.043210,0.092593,0.191358


In [ ]:
df5.to_csv("Virus_season.csv", index=False)


In [ ]:
df5.isnull().sum()

,0
season,0
virus,0
cases_age_0_4,0
cases_age_5_24,0
cases_age_25_64,0
cases_age_65_plus,0
total_cases,0
age_sum,0
diff,0
cases_age_0_4_pct,0


In [ ]:
df.isnull().sum()

,0
region_type,0
region,0
year,0
week,0
ili_weighted_pct,0
ili_unweighted_pct,0
cases_age_0_4,0
cases_age_25_49,0
AGE 25-64,0
cases_age_5_24,0


In [ ]:
df2.isnull().sum()

,0
region_type,0
region,0
year,0
week,0
total_specimens_tested,0
cases_a_h1n1_pdm09,0
cases_a_h3,0
cases_a_not_subtyped,0
cases_b_unspecified,0
cases_b_victoria,0


In [ ]:
df3.isnull().sum()

,0
region_type,0
region,0
year,0
week,0
total_specimens_tested,0
total_influenza_a_cases,0
total_influenza_b_cases,0
percent_positive_overall,0
percent_positive_a,0
percent_positive_b,0


In [ ]:
df4.isnull().sum()

,0
region_type,0
region,0
year,0
week,0
total_specimens_tested,0
percent_positive_overall,0
cases_a_h1n1_2009,0
cases_a_h1,0
cases_a_h3,0
cases_a_not_subtyped,0
